# 复现回测结果

验证本地回测与 `--from-predictions` 结果一致。

In [1]:
import os, sys, pickle, json
import pandas as pd
import numpy as np
sys.path.insert(0, '../code/src')
from backtest import ETFBacktester, run_backtest, run_backtest_from_predictions
import warnings
warnings.filterwarnings('ignore')

## 配置

In [2]:
MODEL_DIR = "../model/search_itransformer_74_3/exp_54"
MODEL_FILE = "best_model_sliding.pth"
DATA_PATH = "../etf_data/etf_74.csv"
CACHE_DIR = "../output/predictions_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

START_DATE = "2026-04-01"
END_DATE = "2026-05-12"
TOP_K = 3
REBALANCE_DAYS = 5
POSITION_PCT = 0.95
INITIAL_CAPITAL = 100000

In [3]:
import glob
from tqdm import tqdm

BASE_DIR = "../model"
MODEL_TYPES = [
                "bayes_itransformer_74_3",
                "search_itransformer_74_3",
                "bayes_dlinear_74_3",
                "bayes_lstm_74_3",
                "bayes_gru_74_3",
                "search_tcn_74_3",
               ]

EXPERIMENTS = []
for mt in MODEL_TYPES:
    cnt = 0
    for exp_dir in sorted(glob.glob(f"{BASE_DIR}/{mt}/exp_*")):
        if os.path.exists(f"{exp_dir}/best_model_sliding.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_sliding.pth"))
        if os.path.exists(f"{exp_dir}/best_model.pth"):
            EXPERIMENTS.append((exp_dir, "best_model.pth"))
        cnt += 1
    print(f"{mt}: 共 {cnt} 个实验，找到 {len(EXPERIMENTS)} 个模型文件")
print(f"共 {len(EXPERIMENTS)} 个实验")

bayes_itransformer_74_3: 共 162 个实验，找到 324 个模型文件
search_itransformer_74_3: 共 72 个实验，找到 468 个模型文件
bayes_dlinear_74_3: 共 0 个实验，找到 468 个模型文件
bayes_lstm_74_3: 共 0 个实验，找到 468 个模型文件
bayes_gru_74_3: 共 105 个实验，找到 676 个模型文件
search_tcn_74_3: 共 54 个实验，找到 782 个模型文件
共 782 个实验


## 遍历所有实验，一次性缓存 + 回测

In [4]:
import time

cached_data, cached_features = ETFBacktester.load_data_once(
    data_path=DATA_PATH,
    scaler_path=f'{MODEL_DIR}/scaler.pkl',
    feature_num='39',
    verbose=True,
)


加载并缓存数据: ../etf_data/etf_74.csv


特征工程: 100%|██████████| 74/74 [00:01<00:00, 38.07it/s]


数据缓存完成: 77996 条记录, 74 只股票


In [5]:


all_results = []
for exp_dir, mf in tqdm(EXPERIMENTS, desc="回测"):
    cache_key = f"{exp_dir}/{mf}"
    safe_name = cache_key.replace("\\", "/").replace('../', '').replace('./', '').replace('/', '_')
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    if not os.path.exists(cache_path):
        try:
            bt = ETFBacktester.from_cached_data(
                model_dir=exp_dir, cached_data=cached_data,
                cached_features=cached_features, device='cpu',
                model_file=mf, verbose=False,
            )
            preds = bt.generate_predictions_dict(start_date=START_DATE, end_date=END_DATE, rebalance_days=REBALANCE_DAYS, first_rebalance_date=START_DATE)
            with open(cache_path, 'wb') as f:
                pickle.dump(preds, f)
            del bt.model, bt
        except Exception as e:
            print(f'FAIL gen {cache_key}: {e}')
            continue
    else:
        with open(cache_path, 'rb') as f:
            preds = pickle.load(f)
    
    for mode in ['close', 'open']:
        try:
            r = run_backtest_from_predictions(
                predictions_dict=preds, data_path=DATA_PATH,
                start_date=START_DATE, end_date=END_DATE,
                top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
                position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
                trade_mode=mode, verbose=False, log=False,
            )
            all_results.append({
                'experiment': exp_dir.replace("\\", "/").split('/')[-2] + '/' + exp_dir.replace("\\", "/").split('/')[-1],
                'model_file': mf, 'trade_mode': mode,
                'return': r.strategy_return,
                'dd': r.max_drawdown,
                'hs300': r.hs300_return,
                'excess': r.excess_return,
                'win_rate': r.rebalance_stats.get('win_rate', 0),
                'avg_return': r.rebalance_stats.get('avg_return', 0),
                'rebalances': r.rebalance_stats.get('total', 0),
            })
        except Exception as e:
            print(f'FAIL backtest {cache_key} {mode}: {e}')

df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('return', ascending=False).head(5)
    print(f'\n=== {mode} Top 5 ===')
    for _, r in sub.iterrows():
        print(f'  {r["experiment"]:35s} {r["model_file"]:25s} return={r["return"]:6.2f}%  dd={r["dd"]:5.2f}%  win={r["win_rate"]:5.1f}%  avg={r["avg_return"]:+5.2f}%')

回测: 100%|██████████| 782/782 [19:27<00:00,  1.49s/it]


=== close Top 5 ===
  bayes_itransformer_74_3/exp_66      best_model_sliding.pth    return= 28.42%  dd= 5.88%  win=100.0%  avg=+4.10%
  bayes_itransformer_74_3/exp_25      best_model_sliding.pth    return= 27.61%  dd= 5.88%  win=100.0%  avg=+4.20%
  bayes_itransformer_74_3/exp_44      best_model_sliding.pth    return= 27.48%  dd= 5.88%  win=100.0%  avg=+3.74%
  search_itransformer_74_3/exp_54     best_model.pth            return= 27.41%  dd= 3.04%  win=100.0%  avg=+3.88%
  search_itransformer_74_3/exp_54     best_model_sliding.pth    return= 27.41%  dd= 3.04%  win=100.0%  avg=+3.88%

=== open Top 5 ===
  bayes_itransformer_74_3/exp_10      best_model_sliding.pth    return= 17.08%  dd= 0.61%  win=100.0%  avg=+2.85%
  bayes_itransformer_74_3/exp_10      best_model.pth            return= 17.08%  dd= 0.61%  win=100.0%  avg=+2.85%
  search_itransformer_74_3/exp_4      best_model_sliding.pth    return= 14.48%  dd= 2.44%  win=100.0%  avg=+2.08%
  search_itransformer_74_3/exp_28     best_mode

In [6]:
df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('avg_return', ascending=False).head(10)
    display(sub[['experiment', 'model_file', 'return', 'avg_return','dd', 'hs300', 'excess']])

,experiment,model_file,return,avg_return,dd,hs300,excess
320,bayes_itransformer_74_3/exp_25,best_model_sliding.pth,27.61,4.20,5.88,9.53,18.08
184,bayes_itransformer_74_3/exp_14,best_model_sliding.pth,26.84,4.19,5.09,9.53,17.31
186,bayes_itransformer_74_3/exp_14,best_model.pth,26.84,4.19,5.09,9.53,17.31
500,bayes_itransformer_74_3/exp_66,best_model_sliding.pth,28.42,4.10,5.88,9.53,18.89
848,search_itransformer_74_3/exp_54,best_model_sliding.pth,27.41,3.88,3.04,9.53,17.88
850,search_itransformer_74_3/exp_54,best_model.pth,27.41,3.88,3.04,9.53,17.88
722,search_itransformer_74_3/exp_25,best_model.pth,27.37,3.87,4.03,9.53,17.84
720,search_itransformer_74_3/exp_25,best_model_sliding.pth,27.37,3.87,4.03,9.53,17.84
310,bayes_itransformer_74_3/exp_22,best_model.pth,26.68,3.76,5.88,9.53,17.15
308,bayes_itransformer_74_3/exp_22,best_model_sliding.pth,26.68,3.76,5.88,9.53,17.15


,experiment,model_file,return,avg_return,dd,hs300,excess
987,bayes_gru_74_3/exp_16,best_model.pth,12.92,3.61,1.43,9.53,3.39
985,bayes_gru_74_3/exp_16,best_model_sliding.pth,12.92,3.61,1.43,9.53,3.39
9,bayes_itransformer_74_3/exp_10,best_model_sliding.pth,17.08,2.85,0.61,9.53,7.55
11,bayes_itransformer_74_3/exp_10,best_model.pth,17.08,2.85,0.61,9.53,7.55
1119,bayes_gru_74_3/exp_46,best_model.pth,13.30,2.69,1.41,9.53,3.77
939,bayes_gru_74_3/exp_0,best_model.pth,7.63,2.55,3.68,9.53,-1.90
937,bayes_gru_74_3/exp_0,best_model_sliding.pth,7.63,2.55,3.68,9.53,-1.90
775,search_itransformer_74_3/exp_37,best_model.pth,13.93,2.29,2.40,9.53,4.40
1309,bayes_gru_74_3/exp_9,best_model_sliding.pth,12.08,2.26,4.24,9.53,2.55
1311,bayes_gru_74_3/exp_9,best_model.pth,12.08,2.26,4.24,9.53,2.55
